# Data operations

In [ ]:
!pip install tensorflow==2.17.0 psycopg2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 601.3/601.3 MB 6.4 MB/s eta 0:00:0000:0100:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 43.9 MB/s eta 0:00:00
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.15.2
    Uninstalling tensorboard-2.15.2:
      Successfully uninstalled tensorboard-2.15.2
  Attempting uninstall: keras
    Found existing installation: keras 2.15.0
    Uninstalling keras-2.15.0:
      Successfully uninstalled keras-2.15.0
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.15.1
    Uninstalling tensorflow-2.15.1:
      Successfully uninstalled tensorflow-2.15.1


In [ ]:
command = f"kubectl get svc | awk '$1 == \"postgres\" {{print $3}}'"

# Execute the command
result = subprocess.run(command, shell=True, capture_output=True, text=True)

# Get the output
ip = result.stdout.strip()

In [2]:
port=5432

In [3]:
user="postgres"
database="postgres"
password="postgres"

In [4]:
print(ip)
print(port)
print(password)

10.111.99.200
5432
postgres


In [6]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import psycopg2

2025-08-19 16:06:11.992227: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-19 16:06:11.994127: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-08-19 16:06:11.997711: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-08-19 16:06:12.005738: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-08-19 16:06:12.018768: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been 

In [7]:
data=keras.datasets.california_housing.load_data(
    version="large", path="california_housing.npz", test_split=0.2, seed=113
)

743530/743530 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step


In [8]:
print(data)

((array([[-118.27  ,   34.09  ,   52.    , ..., 1048.    ,  491.    ,
           3.7847],
       [-118.36  ,   33.96  ,   21.    , ..., 1286.    ,  557.    ,
           2.7284],
       [-122.39  ,   37.76  ,   52.    , ...,  712.    ,  398.    ,
           3.9722],
       ...,
       [-122.34  ,   37.57  ,   52.    , ...,  876.    ,  359.    ,
           8.2598],
       [-122.18  ,   37.89  ,   18.    , ..., 1634.    ,  734.    ,
           8.1489],
       [-118.43  ,   34.2   ,   29.    , ..., 1942.    ,  679.    ,
           3.1118]], dtype=float32), array([252300., 146900., 290900., ..., 500001., 499000., 238100.],
      dtype=float32)), (array([[-118.36  ,   34.08  ,   45.    , ..., 1265.    ,  455.    ,
           3.3864],
       [-120.2   ,   34.63  ,   14.    , ..., 1487.    ,  488.    ,
           4.4519],
       [-121.21  ,   37.81  ,    8.    , ...,  999.    ,  301.    ,
           5.193 ],
       ...,
       [-121.29  ,   37.97  ,   52.    , ..., 1392.    ,  503.    ,
      

In [9]:
import pandas as pd
from tensorflow.keras.datasets import california_housing

def load_data(version="large", path="california_housing.npz", test_split=0.2, seed=113):
    # load dataset
    (x_train, y_train), (x_test, y_test) = california_housing.load_data(
        version=version, path=path, test_split=test_split, seed=seed
    )
    
    # dataset columns
    column_names = ['Longitude', 'Latitude', 'Median_Housing_Age', 'Total_Rooms', 'Total_Bedrooms', 'Population', 'Households', 'Median_Income']
    
    #create panda dataset
    df_train = pd.DataFrame(x_train, columns=column_names)
    df_train['Price'] = y_train
    
    df_test = pd.DataFrame(x_test, columns=column_names)
    df_test['Price'] = y_test
    
    return df_train, df_test
    

In [10]:
df_train, df_test = load_data()

# see the head of the dataframe to see that everything is okay
print(df_train.head())

    Longitude   Latitude  Median_Housing_Age  Total_Rooms  Total_Bedrooms  \
0 -118.269997  34.090000                52.0       2327.0           555.0   
1 -118.360001  33.959999                21.0       1802.0           556.0   
2 -122.389999  37.759998                52.0       1877.0           427.0   
3 -117.949997  33.919998                11.0       3127.0           706.0   
4 -122.519997  37.919998                24.0        421.0            64.0   

   Population  Households  Median_Income     Price  
0      1048.0       491.0         3.7847  252300.0  
1      1286.0       557.0         2.7284  146900.0  
2       712.0       398.0         3.9722  290900.0  
3      1594.0       694.0         4.3426  141300.0  
4       163.0        75.0        14.5833  500001.0  


In [11]:
total_rooms = df_train.iloc[:, 3]  # Total_Rooms
total_bedrooms = df_train.iloc[:, 4]  # Total_Bedrooms
population = df_train.iloc[:, 5]  # Population
households = df_train.iloc[:, 6]  # Households

#New columns
room_per_house = total_rooms / households
bedrooms_ratio = total_bedrooms / total_rooms
people_per_house = population / households
df_train['Room per House'] = room_per_house
df_train['Bedrooms Ratio'] = bedrooms_ratio
df_train['People per House'] = people_per_house
print(df_train.head())

    Longitude   Latitude  Median_Housing_Age  Total_Rooms  Total_Bedrooms  \
0 -118.269997  34.090000                52.0       2327.0           555.0   
1 -118.360001  33.959999                21.0       1802.0           556.0   
2 -122.389999  37.759998                52.0       1877.0           427.0   
3 -117.949997  33.919998                11.0       3127.0           706.0   
4 -122.519997  37.919998                24.0        421.0            64.0   

   Population  Households  Median_Income     Price  Room per House  \
0      1048.0       491.0         3.7847  252300.0        4.739307   
1      1286.0       557.0         2.7284  146900.0        3.235188   
2       712.0       398.0         3.9722  290900.0        4.716080   
3      1594.0       694.0         4.3426  141300.0        4.505764   
4       163.0        75.0        14.5833  500001.0        5.613333   

   Bedrooms Ratio  People per House  
0        0.238505          2.134419  
1        0.308546          2.308797  
2 

In [12]:

# Conexion settings
conn = psycopg2.connect(
    host=ip,
    port=port,
    database=database,
    user=user,
    password=password
)
cur = conn.cursor()

create_table_query = '''
CREATE TABLE  california_housing (
    Longitude FLOAT,
    Latitude FLOAT,
    Median_Housing_Age FLOAT,
    Total_Rooms FLOAT,
    Total_Bedrooms FLOAT,
    Population FLOAT,
    Households FLOAT,
    Median_Income FLOAT,
    Price FLOAT, 
    RoomPerHouse FLOAT,
    BedroomRatio FLOAT,
    PeoplePerHouse FLOAT
    
);
'''
cur.execute(create_table_query)
conn.commit()




In [13]:
for _, row in df_train.iterrows():
    insert_query = '''
    INSERT INTO california_housing (Longitude, Latitude, Median_Housing_Age, Total_Rooms, 
    Total_Bedrooms, Population, Households, Median_Income, Price,RoomPerHouse,BedroomRatio,PeoplePerHouse)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s);
    '''
    cur.execute(insert_query, tuple(row))

# commit the changes
conn.commit()

# close connection
cur.close()
conn.close()

In [14]:

conn = psycopg2.connect(
    host=ip,
    port=port,
    database=database,
    user=user,
    password=password
)


cur = conn.cursor()

# query
cur.execute("SELECT * FROM california_housing")


column_names = [desc[0] for desc in cur.description]
rows = cur.fetchall()


print("Columns:", column_names)

print("Values: ",rows[:1])


# close conexion
cur.close()
conn.close()

Columns: ['longitude', 'latitude', 'median_housing_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'price', 'roomperhouse', 'bedroomratio', 'peopleperhouse']
Values:  [(-118.2699966430664, 34.09000015258789, 52.0, 2327.0, 555.0, 1048.0, 491.0, 3.7846999168395996, 252300.0, 4.739307403564453, 0.23850451409816742, 2.1344194412231445)]


In [ ]:
print("your Data conector: ")
print(f'URI: "jdbc:postgresql://{ip}:{port}/postgres"')
print(f'Username: "{user}"')
print(f'Password: "{password}"')

your data conector: 
URI: "jdbc:postgresql://10.111.99.200:5432/postgres"
Username: "postgres"
Password: "postgres"
